## Hybrid Retriever- Combining Dense And Sparse Retriever

In [4]:
!uv add rank_bm25

Resolved 197 packages in 7.07s
   Building rag @ file:///D:/Mission%20Project/RAG
      Built rag @ file:///D:/Mission%20Project/RAG
Prepared 2 packages in 138ms
Uninstalled 1 package in 2ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 16ms
 ~ rag==0.1.0 (from file:///D:/Mission%20Project/RAG)
 + rank-bm25==0.2.2


In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.schema import Document

In [2]:
## Step 1: Sample documents
docs = [
    Document(page_content="Langchain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for sementic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")

]

## Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

d:\Mission Project\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2791.79it/s]


In [6]:
## Step 3: Sparse Retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k=3 ## top-k documents to retriever

## Step 4: Combine with Ensemble Retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7,0.3]
)

In [7]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001656A02D550>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000165AD0A4550>, k=3)], weights=[0.7, 0.3])

In [ ]:
## Query and get results
query = "How can I build an application using LLM?"
results = hybrid_retriever.invoke(query)

## Print results
for i, doc in enumerate(results):
    print(f"\n Document {i+1}: \n {doc.page_content}")


 Document 1: 
 Langchain helps build LLM applications.

 Document 2: 
 Langchain can be used to develop agentic ai application.

 Document 3: 
 Langchain has many types of retrievers.

 Document 4: 
 Pinecone is a vector database for sementic search.


### RAG Pipeline with hybrid retriever

In [11]:
from langchain_groq import ChatGroq
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [14]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [15]:
## Step 5: Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question based on context below.

Context:
{context}

Question: {input}
""")

## Step 6: llm
llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0.2)

In [16]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000165AD6EC190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000165AD6ECB90>, model_name='openai/gpt-oss-120b', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
## Step 7: Create stuff Document Chain
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

## Step 8: Create Full RAG Chain
rag_chain = create_retrieval_chain(retriever=hybrid_retriever, combine_docs_chain=document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001656A02D550>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000165AD0A4550>, k=3)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')
            | ChatGroq(metadata={'lc

In [20]:
## Step 9: Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

## Step 10: Output
print("Answer:\n", response["answer"])

print("\n Source Documents: ")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

Answer:
 Below is a practical, step‑by‑step outline for building an LLM‑powered application using the tools mentioned in the context (LangChain and Pinecone). You can adapt the flow to the specific use‑case you have in mind (chatbot, question‑answering system, agentic workflow, etc.).

---

## 1. Define the App’s Goal & Architecture
| Step | What to decide |
|------|----------------|
| **Use‑case** | e.g., “semantic search over a knowledge base”, “agent that can browse the web and act on user commands”, “chat assistant that references internal documents”. |
| **LLM provider** | OpenAI, Anthropic, Cohere, etc. (LangChain supports many). |
| **Data source** | PDFs, webpages, database rows, CSVs, etc. |
| **Interaction mode** | Pure retrieval‑augmented generation (RAG), autonomous agent loops, or a hybrid. |

---

## 2. Set Up Your Development Environment
```bash
# Create a virtual env (optional but recommended)
python -m venv venv
source venv/bin/activate   # on Windows: venv\Scripts\act